In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===========================================
# 1. DATA PREPARATION AND SLIDING WINDOW
# ===========================================
print("Data is loaded and a Sliding Window is created...")
df_raw = pd.read_csv('time_series_60min_singleindex.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')
base_features = ['AT_solar_generation_actual', 'AT_wind_onshore_generation_actual', 'AT_load_actual_entsoe_transparency']
data = df_raw[base_features].copy().dropna()

time_features = pd.DataFrame(index=data.index)
time_features['hour'] = time_features.index.hour
time_features['month'] = time_features.index.month
time_features['hour_sin'] = np.sin(2 * np.pi * time_features['hour'] / 24.0)
time_features['hour_cos'] = np.cos(2 * np.pi * time_features['hour'] / 24.0)
time_features['month_sin'] = np.sin(2 * np.pi * time_features['month'] / 12.0)
time_features['month_cos'] = np.cos(2 * np.pi * time_features['month'] / 12.0)
data = pd.concat([data, time_features[['hour_sin', 'hour_cos', 'month_sin', 'month_cos']]], axis=1)

scaler = MinMaxScaler(feature_range=(-1, 1))
data_scaled = scaler.fit_transform(data.values)

def create_sequences(data_array, lookback=48, horizon=24):
    X, y = [], []
    for i in range(len(data_array) - lookback - horizon + 1):
        X.append(data_array[i : (i + lookback), :])
        y.append(data_array[(i + lookback) : (i + lookback + horizon), :].flatten()) 
    return np.array(X), np.array(y)

X, y = create_sequences(data_scaled, 48, 24)

# Training Set
X_train = torch.from_numpy(X[:-3]).float()
y_train = torch.from_numpy(y[:-3]).float()
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=False)

# Forecast Set (Last 3 floating windows reserved for predicting the future)
X_forecast = torch.from_numpy(X[-3:]).float()
y_actual = y[-3:] 
forecast_dates = data.index[-24 * 3:] 

# ===========================================
# 2. ARCHITECTURES AND SMOOTHNESS FUNCTION
# ===========================================
INPUT_SIZE = 7
OUTPUT_SIZE = 7 * 24 

class SpatialSmoothnessLoss(nn.Module):
    def __init__(self, lambda_smooth=0.05):
        super(SpatialSmoothnessLoss, self).__init__()
        self.mse = nn.MSELoss()
        self.lambda_smooth = lambda_smooth
    def forward(self, predictions, targets):
        base_loss = self.mse(predictions, targets)
        preds_reshaped = predictions.view(-1, 24, 7)
        diffs = preds_reshaped[:, 1:, :] - preds_reshaped[:, :-1, :]
        return base_loss + (self.lambda_smooth * torch.mean(diffs ** 2))

class Model_GRU(nn.Module):
    def __init__(self):
        super(Model_GRU, self).__init__()
        self.gru = nn.GRU(input_size=INPUT_SIZE, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, OUTPUT_SIZE)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

class Model_LSTM(nn.Module):
    def __init__(self):
        super(Model_LSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=INPUT_SIZE, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, OUTPUT_SIZE)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class Model_BiLSTM(nn.Module):
    def __init__(self):
        super(Model_BiLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=INPUT_SIZE, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
        self.fc = nn.Linear(128, OUTPUT_SIZE) 
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# ===========================================
# 3. TRAINING, PRINT AND FORECAST CYCLE
# ===========================================
def train_and_forecast_with_metrics(ModelClass, model_name):
    print(f"\n--- {model_name} EĞİTİLİYOR VE FORECAST ALINIYOR ---")
    model = ModelClass().to(device)
    criterion = SpatialSmoothnessLoss(lambda_smooth=0.05)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    for epoch in range(20):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            
    # Forecast Phase
    model.eval()
    with torch.no_grad():
        preds = model(X_forecast.to(device)).cpu().numpy()
        
    # Inverse Scaling and Metric Calculation
    preds_mw = scaler.inverse_transform(preds.reshape(-1, INPUT_SIZE)).reshape(preds.shape)
    y_actual_mw = scaler.inverse_transform(y_actual.reshape(-1, INPUT_SIZE)).reshape(y_actual.shape)

    actual_solar = y_actual_mw[:, 0::INPUT_SIZE]
    actual_wind  = y_actual_mw[:, 1::INPUT_SIZE]
    actual_load  = y_actual_mw[:, 2::INPUT_SIZE]
    
    pred_solar = preds_mw[:, 0::INPUT_SIZE]
    pred_wind  = preds_mw[:, 1::INPUT_SIZE]
    pred_load  = preds_mw[:, 2::INPUT_SIZE]

    # NET LOAD CALCULATION
    actual_net_load = actual_load - (actual_solar + actual_wind)
    pred_net_load = pred_load - (pred_solar + pred_wind)
    
    net_load_mae = mean_absolute_error(actual_net_load, pred_net_load)
    net_load_mse = mean_squared_error(actual_net_load, pred_net_load)
    net_load_rmse = np.sqrt(net_load_mse)
    net_load_wape = (net_load_mae / np.mean(np.abs(actual_net_load))) * 100

    # FLEXIBILITY (RAMPING) CALCULATION
    actual_ramp = np.abs(np.diff(actual_net_load, axis=1))
    pred_ramp = np.abs(np.diff(pred_net_load, axis=1))
    ramp_mae = mean_absolute_error(actual_ramp, pred_ramp)
    
    # SCREEN PRINTING SECTION
    print(f"[{model_name}] FORECAST (GELECEK TAHMİNİ) SONUÇLARI:")
    print(f" -> Net Yük WAPE: %{net_load_wape:.2f} | Net Yük MAE: {net_load_mae:.2f} MW")
    print(f" -> Net Yük MSE:  {net_load_mse:.2f} | Net Yük RMSE: {net_load_rmse:.2f} MW")
    print(f" -> Esneklik (Ramping) MAE: {ramp_mae:.2f} MW")
    
    return pred_load.flatten()

# ===========================================
# 4. CREATING A DATA FRAMEWORK FOR POWER BI
# ===========================================
print("\n===========================================================")
print("FORECAST TEST STARTS (Outputs Will Be Printed on the Screen)")
print("================================================================================")

y_actual_full_mw = scaler.inverse_transform(y_actual.reshape(-1, INPUT_SIZE)).reshape(y_actual.shape)
actual_load_full = y_actual_full_mw[:, 2::INPUT_SIZE].flatten()

df_results = pd.DataFrame({
    'Tarih': forecast_dates,
    'Gercek_Tuketim_MW': actual_load_full,
    'GRU_Tahmin_MW': train_and_forecast_with_metrics(Model_GRU, "GRU"),
    'LSTM_Tahmin_MW': train_and_forecast_with_metrics(Model_LSTM, "LSTM"),
    'BiLSTM_Tahmin_MW': train_and_forecast_with_metrics(Model_BiLSTM, "Bi-LSTM")
})

# Adding Error Rates to the Table
df_results['GRU_Hata'] = df_results['Gercek_Tuketim_MW'] - df_results['GRU_Tahmin_MW']
df_results['LSTM_Hata'] = df_results['Gercek_Tuketim_MW'] - df_results['LSTM_Tahmin_MW']
df_results['BiLSTM_Hata'] = df_results['Gercek_Tuketim_MW'] - df_results['BiLSTM_Tahmin_MW']

df_results.to_csv('power_bi_forecast_sonuclari.csv', index=False)
print("\n===========================================================")
print("All models have been tested successfully! The file 'power_bi_forecast_sonuclari.csv' has been created.")